In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import pandas as pd
import torch
import gc
from transformers import pipeline

# 1. Automated Path Discovery
base_path = "/kaggle/input"
target_file = "product_descriptions.csv"
file_path = ""

for root, dirs, files in os.walk(base_path):
    if target_file in files:
        file_path = os.path.join(root, target_file)
        break

if file_path:
    df = pd.read_csv(file_path)
    print(f"Successfully loaded {len(df)} products from {file_path}")
else:
    raise FileNotFoundError("Could not find product_descriptions.csv. Please check your Input sidebar.")

# 2. Load LLM for Prompt Engineering
llm_pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", torch_dtype=torch.float16, device=0)

def rewrite_prompt(description):
    messages = [
        {"role": "system", "content": "You are an expert prompt engineer for Stable Diffusion. Convert product descriptions into professional photography prompts. Include: lighting (rim lighting, softbox), camera (85mm lens, f/1.8), and quality (ultra-realistic, 8k, highly detailed). Keep it under 60 words."},
        {"role": "user", "content": f"Create a professional image prompt for: {description}"}
    ]
    
    output = llm_pipe(messages, max_new_tokens=80, do_sample=True, temperature=0.7)
    engineered_prompt = output[0]['generated_text'][-1]['content']
    
    # Clean up to ensure it stays under the 77-token CLIP limit
    return engineered_prompt.strip()[:300]

df['engineered_prompt'] = df['product_description'].apply(rewrite_prompt)

# 3. Clear VRAM for Stable Diffusion
del llm_pipe
gc.collect()
torch.cuda.empty_cache()
print("Prompts engineered. Memory cleared for Stable Diffusion.")

C:\Users\pardhu\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully loaded 15 products from .\product_descriptions.csv


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 613.30it/s, Materializing param=model.norm.weight]                              
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` 

Prompts engineered. Memory cleared for Stable Diffusion.


In [ ]:
from diffusers import StableDiffusionPipeline

# Load Stable Diffusion v1.5 with float16 for memory efficiency
device = 0 if torch.cuda.is_available() else -1
sd_pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
).to(device)

def generate_image(prompt):
    negative_prompt = "blurry, distorted, low quality, text, watermark, deformed, messy, grainy, low resolution"
    
    image = sd_pipe(
        prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=20,  # Reduced for faster generation
        guidance_scale=7.5
    ).images[0]
    
    return image

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.
Loading weights: 100%|██████████| 396/396 [00:00<00:00, 520.55it/s, Materializing param=visual_projection.weight]
StableDiffusionSafetyChecker LOAD REPORT from: C:\Users\pardhu\.cache\huggingface\hub\models--runwayml--stable-diffusion-v1-5\snapshots\451f4fe16113bff5a5d2269ed5ad43b0592e9a14\safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 196/196 [00:00<00:00, 526.46it/

In [5]:
import gradio as gr

def process_and_generate(description):
    new_prompt = rewrite_prompt(description)
    img = generate_image(new_prompt)
    return new_prompt, img

# Build the Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎨 NIAT AI Product Image Generator")
    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(label="Product Description", placeholder="e.g., Red wireless headphones")
            generate_btn = gr.Button("Generate Professional Image", variant="primary")
        with gr.Column():
            output_prompt = gr.Textbox(label="AI Engineered Prompt")
            output_image = gr.Image(label="Final Product Image")
            
    generate_btn.click(fn=process_and_generate, inputs=input_text, outputs=[output_prompt, output_image])

# Launch for the assignment
demo.launch(share=True)

C:\Users\pardhu\AppData\Local\Temp\ipykernel_43528\543575366.py:9: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


In [ ]:
import pandas as pd

# Load the 15 descriptions
df = pd.read_csv("product_descriptions.csv")
results = []

for index, row in df.iterrows():
    print(f"Processing {index+1}/15: {row['product_description']}")
    eng_prompt = rewrite_prompt(row['product_description'])
    img = generate_image(eng_prompt)
    
    # Save the image locally as required by the starter notebook
    img_path = f"product_{row['id']}.png"
    img.save(img_path)
    
    results.append({
        "id": row['id'],
        "engineered_prompt": eng_prompt,
        "image_path": img_path
    })

# Convert to final results dataframe
final_df = pd.DataFrame(results)
final_df.to_csv("submission_results.csv", index=False)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/product-image-generator/product_descriptions.csv'

In [ ]:
import gc
del llm_pipe
torch.cuda.empty_cache()
gc.collect()